# Class 7 — Per-Account Behavioural Baselines

**Deliverable:** Per-entity rolling statistics + velocity features
**Date:** August 2, 2026

Up to Class 6 the z-score was **global**: an amount was judged unusual relative to all
284,807 transactions in the dataset. That is the wrong comparison.

A £900 charge is unremarkable for a business account that spends £900 every week, and
highly suspicious for a student who has never spent more than £40. Judging both against
the same mean discards the only context that makes the number mean anything.

The project proposal specifies *"rolling statistics per account/user stored in a
lightweight database"* and *"statistical baselining (rolling mean/standard deviation,
z-scores) for per-entity behaviour."* This class implements it.

## The streaming constraint

In a notebook you can write `df.groupby('account')['Amount'].mean()`. In a live pipeline
you cannot:

- transactions arrive one at a time
- you cannot hold every past transaction in memory
- you must answer in milliseconds

So the statistics must be computed **incrementally** — each new transaction updates a
fixed-size summary without revisiting history.

In [ ]:
import sys, math, random, statistics, time
sys.path.insert(0, '../src')

from baseline_store import AccountState, BaselineStore, combine_severity, MIN_HISTORY, EWMA_ALPHA

print('MIN_HISTORY =', MIN_HISTORY, ' EWMA_ALPHA =', EWMA_ALPHA)

## Part 1 — Why the obvious formula is wrong

The textbook streaming variance is $\mathrm{Var}(x) = E[x^2] - (E[x])^2$. It only needs
a running sum and a running sum of squares, so it looks perfect for streaming.

It is numerically unstable. It subtracts two large, nearly equal floating point numbers,
and the significant digits cancel. On realistic magnitudes it can return a **negative
variance** — and `sqrt` of a negative is NaN.

In [ ]:
data = [1e9 + i * 0.001 for i in range(1000)]

n = len(data)
naive_var = sum(x*x for x in data)/n - (sum(data)/n)**2
exact_var = statistics.variance(data)   # exact Fraction arithmetic

print(f'exact variance : {exact_var:.6e}')
print(f'naive variance : {naive_var:.6e}   <-- negative!')

### Welford's algorithm (1962)

Accumulate deviations from a *running* mean instead:

```
n    += 1
delta = x - mean
mean += delta / n
M2   += delta * (x - mean)     # note: the UPDATED mean
```

Using the updated mean in the second line is what makes it stable. `M2 / (n-1)` is the
sample variance. Memory is O(1) — three numbers, regardless of history length.

In [ ]:
st = AccountState('demo')
for i, x in enumerate(data):
    st.update(x, ts=float(i))

print(f'welford variance: {st.variance:.6e}')
print()
print(f'naive   relative error: {abs(naive_var-exact_var)/exact_var:.2e}')
print(f'welford relative error: {abs(st.variance-exact_var)/exact_var:.2e}')

Welford's residual error is float64 input precision — `1e9 + 0.001` is not exactly
representable, so the *stored data* is already approximate. The algorithm is not adding
error, it is just not amplifying it.

### Does it match batch statistics on realistic data?

In [ ]:
random.seed(7)
sample = [random.lognormvariate(4, 1.1) for _ in range(5000)]

st = AccountState('check')
for i, x in enumerate(sample):
    st.update(x, ts=float(i))

print(f'{"":10} {"online":>14} {"batch":>14}')
print(f'{"mean":10} {st.mean:>14.9f} {statistics.fmean(sample):>14.9f}')
print(f'{"std":10} {st.std:>14.9f} {statistics.stdev(sample):>14.9f}')
print()
print('identical to 9 decimal places, using 3 numbers instead of 5000')

## Part 2 — Cumulative vs recent: the EWMA

Welford gives the **lifetime** mean. But accounts genuinely change — someone gets a
raise, a business grows. A lifetime average from three years ago is not the right
baseline today.

The exponentially weighted moving average weights recent observations more heavily:

$$\mu_t = \alpha x_t + (1-\alpha)\mu_{t-1}$$

With $\alpha = 0.05$ the effective window is about $1/\alpha = 20$ transactions.

Both are kept, because the **gap between them** is itself a signal: a large divergence
means the account's behaviour is drifting, not that one transaction is odd.

In [ ]:
st = AccountState('drift')
for i in range(500):            # settled around 50
    st.update(50.0, ts=float(i))

print(f'after 500 txns at 50:   cumulative={st.mean:7.2f}   ewma={st.ewma_mean:7.2f}')

for i in range(500, 700):       # account moves to 500
    st.update(500.0, ts=float(i))

print(f'after 200 txns at 500:  cumulative={st.mean:7.2f}   ewma={st.ewma_mean:7.2f}')
print()
print('EWMA has followed the change; the cumulative mean is still dragged')
print('down by 500 old observations. Judging today against the cumulative')
print('mean would flag every normal transaction from now on.')

## Part 3 — The cold start problem

A brand new account has no history. With `n=3`, one large purchase makes the standard
deviation enormous and nothing ever looks anomalous again. With `n=1`, variance is
undefined entirely.

So below `MIN_HISTORY = 30` observations, the store refuses to offer a per-account
z-score and the caller falls back to the global baseline from Class 6. **This is why the
global z-score is kept rather than replaced.**

In [ ]:
store = BaselineStore(':memory:', global_mean=88.29, global_std=250.12)

for n in [1, 5, 10, 29, 30, 31]:
    s = BaselineStore(':memory:', global_mean=88.29, global_std=250.12)
    for i in range(n):
        s.observe('NEW', 100.0 + i, ts=float(i*60))
    f = s.features('NEW', 500.0, ts=99999.0)
    az = f['account_z'] if f['account_z'] is not None else '-- falls back to global --'
    print(f'  n={n:3}  warm={str(f["account_warm"]):5}  account_z={az}')
    s.close()

## Part 4 — The motivating example

Same amount, two different accounts. This is the whole argument for the class.

In [ ]:
random.seed(1)
s = BaselineStore(':memory:', global_mean=88.29, global_std=250.12)

for i in range(200):
    s.observe('BIZ', random.gauss(900, 80), ts=float(i*3600))   # business
for i in range(200):
    s.observe('STU', random.gauss(25, 6),   ts=float(i*3600))   # student

amount, ts = 900.0, float(300*3600)
biz = s.features('BIZ', amount, ts)
stu = s.features('STU', amount, ts)

print(f'A {amount:.0f} transaction:\n')
print(f'{"":18} {"global z":>10} {"account z":>11} {"acct mean":>11}  verdict')
for name, f in [('business acct', biz), ('student acct', stu)]:
    verdict = 'FLAG' if f['account_z'] > 3 else 'normal'
    print(f'{name:18} {f["global_z"]:>10.2f} {f["account_z"]:>11.2f} '
          f'{f["account_mean"]:>11.2f}  {verdict}')

print('\nThe global z-score is identical for both - it cannot tell them apart.')
s.close()

## Part 5 — Velocity: what amount checks cannot see

**Card testing:** a stolen card is validated with many tiny charges before the real
fraud. Every individual charge looks innocent — that is the point of the attack.

Catching it requires per-account *state across transactions*, which is only possible now
that we track accounts.

In [ ]:
random.seed(2)
s = BaselineStore(':memory:', global_mean=88.29, global_std=250.12)

for i in range(200):                       # normal history: ~60 per day
    s.observe('VICTIM', random.gauss(60, 10), ts=float(i*86400))

base = float(200*86400)
for i in range(20):                        # 20 charges of 3.00 in 10 minutes
    f = s.observe('VICTIM', 3.00, ts=base + i*30)

sev, reasons = combine_severity(
    account_z=f['account_z'], global_z=f['global_z'],
    txn_count_1h=f['txn_count_1h'], ml_flagged=False,
    account_warm=f['account_warm'])

print(f'amount             : 3.00')
print(f'account_z          : {f["account_z"]:.2f}   <- looks totally innocent')
print(f'global_z           : {f["global_z"]:.2f}   <- also innocent')
print(f'txn_count_1h       : {f["txn_count_1h"]}      <- this is the signal')
print(f'seconds_since_last : {f["seconds_since_last"]}')
print()
print(f'severity : {sev}')
for r in reasons:
    print(f'   - {r}')
s.close()

## Part 6 — What it costs

A baseline layer that is accurate but slow is useless in a real-time pipeline. Measured
on 50,000 transactions across 1,000 accounts:

In [ ]:
from simulate_stream import benchmark
benchmark('/tmp/class7_bench.db')

Roughly **0.011 ms per transaction** and **~127 bytes per account**. For comparison,
the Isolation Forest inference in `scoring_service.py` takes on the order of a
millisecond, so the baseline layer adds around 1% to the scoring cost.

Storage is O(accounts), not O(transactions) — 1 million customers would need about
127 MB regardless of how long they have been customers. That is the practical payoff of
the O(1) update: history is summarised, never stored.

## Part 7 — Running it on the real dataset

`creditcard.csv` has **no account column**. Every row is anonymous — which is precisely
why Classes 2–6 had to use a global z-score.

Per-account baselining cannot be demonstrated on data with no accounts, so
`simulate_stream.py` assigns synthetic account IDs.

> **This is synthetic and must be stated as such in the report.**
> It demonstrates that the machinery works and what it costs. It does **not**
> demonstrate improved fraud recall on real per-account data, and no such claim
> should be made from it.

Two modes are provided deliberately:

| Mode | What it does | What it shows |
|---|---|---|
| `random` | accounts drawn uniformly | the **null case** — when accounts are interchangeable, per-account baselines add nothing |
| `profiled` | accounts given distinct spend levels | the mechanism working — but the data was *constructed* to have the property being measured |

Reporting only `profiled` would be dishonest. Run both.

In [ ]:
# Requires ../data/creditcard.csv — see data/README.md
from pathlib import Path
from simulate_stream import run

if Path('../data/creditcard.csv').exists():
    for mode in ['random', 'profiled']:
        print('=' * 60)
        out = run('../data/creditcard.csv', rows=50_000, n_accounts=500,
                  mode=mode, db_path=f'/tmp/sim_{mode}.db')
        print(f'\n  {mode} summary:')
        for k, v in out.items():
            print(f'    {k:22} {v}')
        print()
else:
    print('creditcard.csv not found — see data/README.md')

## What I learned

- **Welford's algorithm** — exact streaming mean and variance in O(1) memory, and why
  the obvious $E[x^2]-E[x]^2$ formula fails on real magnitudes
- **EWMA** — adapting a baseline to genuine behavioural change, and using the gap
  between cumulative and recent means as a drift signal
- **The cold start problem** — why a new account cannot be judged against itself, and
  why the global baseline from Class 6 has to stay as a fallback
- **Ordering matters** — a transaction must be scored against history that excludes
  itself, or a large fraudulent charge partly normalises its own baseline
- **Velocity features** — stateful signals that no single-transaction check can see
- **Honest evaluation** — when the data has to be synthesised to demonstrate a method,
  the demonstration is not evidence, and the write-up has to say so

## Limitations

1. Account IDs are synthetic. Real validation needs a dataset with genuine account
   identifiers — PaySim or IEEE-CIS both have them.
2. `MIN_HISTORY = 30` and `EWMA_ALPHA = 0.05` were chosen by reasoning, not tuned.
3. SQLite is single-writer. Fine for this project; a multi-worker deployment would
   need Redis or Postgres.
4. No account-level seasonality — a monthly salary deposit will always look anomalous.